In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/cleaned_leads.csv")
print("Shape:", df.shape)
df.head()

Shape: (9240, 28)


,Lead Origin,Lead Source,Do Not Email,Do Not Call,Converted,TotalVisits,Total Time Spent on Website,Page Views Per Visit,Last Activity,Country,...,Through Recommendations,Receive More Updates About Our Courses,Tags,Lead Quality,Update me on Supply Chain Content,Get updates on DM Content,City,I agree to pay the amount through cheque,A free copy of Mastering The Interview,Last Notable Activity
0,API,Olark Chat,No,No,0,0.0,0,0.0,Page Visited on Website,Unknown,...,No,No,Interested in other courses,Low in Relevance,No,No,Unknown,No,No,Modified
1,API,Organic Search,No,No,0,5.0,674,2.5,Email Opened,India,...,No,No,Ringing,Unknown,No,No,Unknown,No,No,Email Opened
2,Landing Page Submission,Direct Traffic,No,No,1,2.0,1532,2.0,Email Opened,India,...,No,No,Will revert after reading the email,Might be,No,No,Mumbai,No,Yes,Email Opened
3,Landing Page Submission,Direct Traffic,No,No,0,1.0,305,1.0,Unreachable,India,...,No,No,Ringing,Not Sure,No,No,Mumbai,No,No,Modified
4,Landing Page Submission,Google,No,No,1,2.0,1428,1.0,Converted to Lead,India,...,No,No,Will revert after reading the email,Might be,No,No,Mumbai,No,No,Modified


In [2]:
binary_cols = [
    'Do Not Email', 'Do Not Call', 'Search', 'Newspaper Article',
    'X Education Forums', 'Newspaper', 'Digital Advertisement',
    'Through Recommendations', 'Receive More Updates About Our Courses',
    'Update me on Supply Chain Content', 'Get updates on DM Content',
    'I agree to pay the amount through cheque',
    'A free copy of Mastering The Interview'
]

# Check unique values before converting (safety check)
for col in binary_cols:
    print(col, ":", df[col].unique())

Do Not Email : <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Do Not Call : <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Search : <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Newspaper Article : <StringArray>
['No', 'Yes']
Length: 2, dtype: str
X Education Forums : <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Newspaper : <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Digital Advertisement : <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Through Recommendations : <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Receive More Updates About Our Courses : <StringArray>
['No']
Length: 1, dtype: str
Update me on Supply Chain Content : <StringArray>
['No']
Length: 1, dtype: str
Get updates on DM Content : <StringArray>
['No']
Length: 1, dtype: str
I agree to pay the amount through cheque : <StringArray>
['No']
Length: 1, dtype: str
A free copy of Mastering The Interview : <StringArray>
['No', 'Yes']
Length: 2, dtype: str


In [3]:
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# Verify conversion
df[binary_cols].head()

,Do Not Email,Do Not Call,Search,Newspaper Article,X Education Forums,Newspaper,Digital Advertisement,Through Recommendations,Receive More Updates About Our Courses,Update me on Supply Chain Content,Get updates on DM Content,I agree to pay the amount through cheque,A free copy of Mastering The Interview
0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,1
3,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0


In [4]:
# Remaining categorical columns (excluding target)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns to encode:", len(categorical_cols))
print(categorical_cols)

Categorical columns to encode: 11
['Lead Origin', 'Lead Source', 'Last Activity', 'Country', 'Specialization', 'What is your current occupation', 'What matters most to you in choosing a course', 'Tags', 'Lead Quality', 'City', 'Last Notable Activity']


In [5]:
for col in categorical_cols:
    print(f"{col}: {df[col].nunique()} unique values")

Lead Origin: 5 unique values
Lead Source: 21 unique values
Last Activity: 17 unique values
Country: 39 unique values
Specialization: 19 unique values
What is your current occupation: 7 unique values
What matters most to you in choosing a course: 4 unique values
Tags: 27 unique values
Lead Quality: 6 unique values
City: 7 unique values
Last Notable Activity: 16 unique values


In [6]:
high_cardinality_cols = ['Country', 'Tags', 'Lead Source', 'Specialization', 
                          'Last Activity', 'Last Notable Activity']

def group_rare_categories(df, col, threshold=0.01):
    """Group categories that appear in less than `threshold` fraction of rows into 'Other'"""
    freq = df[col].value_counts(normalize=True)
    rare_categories = freq[freq < threshold].index
    df[col] = df[col].replace(rare_categories, 'Other')
    return df

for col in high_cardinality_cols:
    df = group_rare_categories(df, col, threshold=0.01)  # groups categories under 1% of data

# Check new unique counts
for col in high_cardinality_cols:
    print(f"{col}: {df[col].nunique()} unique values (after grouping)")

Country: 3 unique values (after grouping)
Tags: 13 unique values (after grouping)
Lead Source: 8 unique values (after grouping)
Specialization: 17 unique values (after grouping)
Last Activity: 10 unique values (after grouping)
Last Notable Activity: 7 unique values (after grouping)


In [7]:
# Get updated list of categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print("Columns to encode:", categorical_cols)

# One-hot encode
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("\nShape before encoding:", df.shape)
print("Shape after encoding:", df_encoded.shape)

Columns to encode: ['Lead Origin', 'Lead Source', 'Last Activity', 'Country', 'Specialization', 'What is your current occupation', 'What matters most to you in choosing a course', 'Tags', 'Lead Quality', 'City', 'Last Notable Activity']

Shape before encoding: (9240, 28)
Shape after encoding: (9240, 93)


In [8]:
print("Remaining categorical (object) columns:", df_encoded.select_dtypes(include=['object']).columns.tolist())
df_encoded.dtypes.value_counts()

Remaining categorical (object) columns: []


bool       76
int64      15
float64     2
Name: count, dtype: int64

In [9]:
df_encoded.to_csv("../data/processed/featured_leads.csv", index=False)
print("✅ Feature-engineered dataset saved!")
print("Final shape:", df_encoded.shape)

✅ Feature-engineered dataset saved!
Final shape: (9240, 93)
